In [12]:
import pandas as pd
import numpy as np
import requests

In [40]:
df = pd.read_excel(r'C:\Users\ajpk\Desktop\495\mdcr-capstone\data\Contractor C Workforce Data Form.xlsx', skiprows=7, sheet_name=None)

In [41]:
previous_year_df = df['Workforce Data Previous Year']
current_year_df = df['Workforce Data Current Year']

In [42]:
previous_year_df.columns = ['Job Role', 'Hispanic/Latino Male', 'Hispanic/Latina Female', 'White Male', 'Black or African American Male',
    'Asian Male', 'Native Hawaiian or Other Pacific Islander Male',
    'American Indian or Alaskan Native Male', 'Two or more races Male', 'White Female',
    'Black or African American Female', 'Asian Female',
    'Native Hawaiian or Other Pacific Islander Female',
    'American Indian or Alaskan Native Female', 'Two or more races Female',
    'Row Total']
previous_year_df = previous_year_df.iloc[1:13].fillna(0)

current_year_df.columns = ['Job Role', 'Hispanic/Latino Male', 'Hispanic/Latina Female', 'White Male', 'Black or African American Male',
    'Asian Male', 'Native Hawaiian or Other Pacific Islander Male',
    'American Indian or Alaskan Native Male', 'Two or more races Male', 'White Female',
    'Black or African American Female', 'Asian Female',
    'Native Hawaiian or Other Pacific Islander Female',
    'American Indian or Alaskan Native Female', 'Two or more races Female',
    'Row Total']
current_year_df = current_year_df.iloc[1:13].fillna(0)

In [61]:
diff_dict

NameError: name 'diff_dict' is not defined

In [43]:
def census_loader():
	"""
	Pulls and returns regional EEO data from the US Census API

	region: str
	 - Region to pull data from
	 - use "us" for United States
	 - for other states, consult the STATE_TO_FIPS dictionary

	returns: constructed dataframe
	"""

	dataset_url = "https://api.census.gov/data/2018/acs/acs5/eeo" # 2018
	api_key = "32685a5c4ce6a6c58598b54cce7cf23a6608f8f0"

	metadata_url = f"{dataset_url}/variables.json"
	meta_response = requests.get(metadata_url)
	variables_meta = meta_response.json()["variables"]
	var_label_map = {
		var_id: meta["label"].replace("!!", " ")
		for var_id, meta in variables_meta.items()
		if "label" in meta
	}

	# params for loading
	# for: state:26 is Michigan
	# do: for: us-1 for USA
	params = {
		"get": "group(EEOALL1R)",
		"EEO": "0000",
		"ucgid": "0400000US26",
		"key": api_key
	}
	data_response = requests.get(dataset_url, params=params)
	raw_data = data_response.json()

	df = pd.DataFrame(raw_data[1:], columns=raw_data[0])

	df = df.rename(columns=var_label_map)

	return df

In [44]:
cen_df = census_loader()

In [20]:
def calculate_region_ratios(df):
	"""
	Takes regional demographic data and compiles ratios for each group

	df: dataframe
	 - region demographic data

	returns: dictionary {demographic:ratio}
	"""

	categories = [
	"Total Pop",
	"Total Male",
	"Total Female",
	"Hispanic/Latino Pop",
	"Hispanic/Latino Male",
	"Hispanic/Latina Female",
	"White Pop",
	"White Male",
	"White Female",
	"Black or African American Pop",
	"Black or African American Male",
	"Black or African American Female",
	"Native Am Pop",
	"Native Am Male",
	"Native Am Female",
	"Asian Pop",
	"Asian Male",
	"Asian Female",
	"Pacific Pop",
	"Pacific Male",
	"Pacific Female",
	"Other Pop",
	"Other Male",
	"Other Female"]

	df = df.dropna(axis = 1) # percentage is already calculated, just need to extract
	df = df.loc[:, df.columns.str.contains('Percent')]
	
	vals = []
	for i in df.columns: 
		vals.append(df[i].item()) # list of ratios

	region_ratio = np.column_stack((categories, vals)) # stack categories with their respective ratios
	region_ratio_dict = dict(region_ratio) # saves as dictionary for comparison

	return region_ratio_dict

In [21]:
cen_df.columns

Index(['Race/Ethnic Group', 'EEO_LABEL',
       'Estimate Total, race and ethnicity Total, both sexes Number',
       'EEOALL1R_C01_001EA', 'EEOALL1R_C01_001M', 'EEOALL1R_C01_001MA',
       'Estimate Total, race and ethnicity Total, both sexes Percent',
       'EEOALL1R_C01_002EA', 'EEOALL1R_C01_002M', 'EEOALL1R_C01_002MA',
       ...
       'EEOALL1R_C08_005M', 'EEOALL1R_C08_005MA',
       'Estimate Balance of not Hispanic or Latino Total, both sexes Female Percent',
       'EEOALL1R_C08_006EA', 'EEOALL1R_C08_006M', 'EEOALL1R_C08_006MA',
       'Geography', 'NAME', 'Race/Ethnic Group',
       'Uniform Census Geography Identifier clause'],
      dtype='str', length=198)

In [23]:
for col in cen_df.columns:
    print(col)

Race/Ethnic Group
EEO_LABEL
Estimate Total, race and ethnicity Total, both sexes Number
EEOALL1R_C01_001EA
EEOALL1R_C01_001M
EEOALL1R_C01_001MA
Estimate Total, race and ethnicity Total, both sexes Percent
EEOALL1R_C01_002EA
EEOALL1R_C01_002M
EEOALL1R_C01_002MA
Estimate Total, race and ethnicity Total, both sexes Male Number
EEOALL1R_C01_003EA
EEOALL1R_C01_003M
EEOALL1R_C01_003MA
Estimate Total, race and ethnicity Total, both sexes Male Percent
EEOALL1R_C01_004EA
EEOALL1R_C01_004M
EEOALL1R_C01_004MA
Estimate Total, race and ethnicity Total, both sexes Female Number
EEOALL1R_C01_005EA
EEOALL1R_C01_005M
EEOALL1R_C01_005MA
Estimate Total, race and ethnicity Total, both sexes Female Percent
EEOALL1R_C01_006EA
EEOALL1R_C01_006M
EEOALL1R_C01_006MA
Estimate Hispanic or Latino Total, both sexes Number
EEOALL1R_C02_001EA
EEOALL1R_C02_001M
EEOALL1R_C02_001MA
Estimate Hispanic or Latino Total, both sexes Percent
EEOALL1R_C02_002EA
EEOALL1R_C02_002M
EEOALL1R_C02_002MA
Estimate Hispanic or Latino To

In [27]:
t = calculate_region_ratios(cen_df)

In [29]:
def calculate_differences(regional_df, contractor_df):
	"""

	Finds differences of demographic ratio data between region selected and contractor

	regional_df: dataframe
	 - region demographic data

	contractor_df: dataframe
	 - contractor demographic data

	returns: list (demographic,difference)
	(positive value is over-represented, negative value is underrepresented)

	"""

	regional_dict = calculate_region_ratios(regional_df) # get regional demographic ratios
	contractor_dict = calculate_contractor_ratios(contractor_df) # get contractor demographic ratios

	result = []

    regional_dict["Native Hawaiian or Other Pacific Islander Female"] = regional_dict.pop("Pacific Female")
    regional_dict["American Indian or Alaskan Native Female"] = regional_dict.pop("Native Am Female")
    regional_dict["Native Hawaiian or Other Pacific Islander Male"] = regional_dict.pop("Pacific Male")
    regional_dict["American Indian or Alaskan Native Male"] = regional_dict.pop("Native Am Male")
    regional_dict["Two or more races Male"] = regional_dict.pop("Other Male")
    regional_dict["Two or more races Female"] = regional_dict.pop("Other Female")

	for key, val in contractor_dict.items():
		if key in regional_dict:
			result.append([key, round(float(val), 2) - round(float(regional_dict[key]),2)])

	return result, regional_dict, contractor_dict

In [33]:
def calculate_contractor_ratios(df):
	"""
	Takes contractor demographic data and compiles ratios for each group

	df: dataframe
	 - contractor demographic data
	 - 

	returns: dictionary {demographic:ratio}
	"""

	#Labels for each ratio
	updated_categories = ['Hispanic/Latino Male', 'Hispanic/Latina Female',
	   'White Male', 'Black or African American Male', 'Asian Male',
	   'Native Hawaiian or Other Pacific Islander Male',
	   'American Indian or Alaskan Native Male', 'Two or more races Male',
	   'White Female', 'Black or African American Female', 'Asian Female',
	   'Native Hawaiian or Other Pacific Islander Female',
	   'American Indian or Alaskan Native Female', 'Two or more races Female']

	vals = []

	for i in updated_categories: # for each identity
		a = df[i][11] / df["Row Total"][11] * 100 # find percentage relative to total worker count
		b = round(a, 2)
		vals.append(b)

	contractor_ratio = np.column_stack((updated_categories, vals)) # stack the categories and ratios
	contractor_ratio_dict = dict(contractor_ratio) # convert into a dictionary for comparison 
	
	return contractor_ratio_dict

In [35]:
b = calculate_contractor_ratios(current_year_df)

In [54]:
result = []

for key, val in t.items():
    if key in b:
        result.append([key, round(float(val), 2) - round(float(t[key]),2)])

In [55]:
result

[[np.str_('Hispanic/Latino Male'), 0.0],
 [np.str_('Hispanic/Latina Female'), 0.0],
 [np.str_('White Male'), 0.0],
 [np.str_('White Female'), 0.0],
 [np.str_('Black or African American Male'), 0.0],
 [np.str_('Black or African American Female'), 0.0],
 [np.str_('Asian Male'), 0.0],
 [np.str_('Asian Female'), 0.0],
 ['Native Hawaiian or Other Pacific Islander Female', 0.0],
 ['American Indian or Alaskan Native Female', 0.0],
 ['Native Hawaiian or Other Pacific Islander Male', 0.0],
 ['American Indian or Alaskan Native Male', 0.0],
 ['Two or more races Male', 0.0],
 ['Two or more races Female', 0.0]]

In [53]:
t["Native Hawaiian or Other Pacific Islander Female"] = t.pop("Pacific Female")
t["American Indian or Alaskan Native Female"] = t.pop("Native Am Female")
t["Native Hawaiian or Other Pacific Islander Male"] = t.pop("Pacific Male")
t["American Indian or Alaskan Native Male"] = t.pop("Native Am Male")
t["Two or more races Male"] = t.pop("Other Male")
t["Two or more races Female"] = t.pop("Other Female")

In [50]:
t

{np.str_('Total Pop'): np.str_('100.0'),
 np.str_('Total Male'): np.str_('52.3'),
 np.str_('Total Female'): np.str_('47.7'),
 np.str_('Hispanic/Latino Pop'): np.str_('4.6'),
 np.str_('Hispanic/Latino Male'): np.str_('2.6'),
 np.str_('Hispanic/Latina Female'): np.str_('2.1'),
 np.str_('White Pop'): np.str_('77.2'),
 np.str_('White Male'): np.str_('41.1'),
 np.str_('White Female'): np.str_('36.2'),
 np.str_('Black or African American Pop'): np.str_('12.5'),
 np.str_('Black or African American Male'): np.str_('5.7'),
 np.str_('Black or African American Female'): np.str_('6.8'),
 np.str_('Native Am Pop'): np.str_('0.4'),
 np.str_('Native Am Male'): np.str_('0.2'),
 np.str_('Native Am Female'): np.str_('0.2'),
 np.str_('Asian Pop'): np.str_('3.2'),
 np.str_('Asian Male'): np.str_('1.8'),
 np.str_('Asian Female'): np.str_('1.4'),
 np.str_('Pacific Pop'): np.str_('0.0'),
 np.str_('Pacific Male'): np.str_('0.0'),
 np.str_('Pacific Female'): np.str_('0.0'),
 np.str_('Other Pop'): np.str_('1.9')

In [51]:
b

{np.str_('Hispanic/Latino Male'): np.str_('2.61'),
 np.str_('Hispanic/Latina Female'): np.str_('3.51'),
 np.str_('White Male'): np.str_('32.82'),
 np.str_('Black or African American Male'): np.str_('3.33'),
 np.str_('Asian Male'): np.str_('5.11'),
 np.str_('Native Hawaiian or Other Pacific Islander Male'): np.str_('0.06'),
 np.str_('American Indian or Alaskan Native Male'): np.str_('0.2'),
 np.str_('Two or more races Male'): np.str_('0.34'),
 np.str_('White Female'): np.str_('42.33'),
 np.str_('Black or African American Female'): np.str_('3.94'),
 np.str_('Asian Female'): np.str_('4.66'),
 np.str_('Native Hawaiian or Other Pacific Islander Female'): np.str_('0.05'),
 np.str_('American Indian or Alaskan Native Female'): np.str_('0.26'),
 np.str_('Two or more races Female'): np.str_('0.77')}

In [69]:
pd.DataFrame(result, columns=['Demographic', 'Percent Difference'])

,Demographic,Percent Difference
0,Hispanic/Latino Male,0.0
1,Hispanic/Latina Female,0.0
2,White Male,0.0
3,White Female,0.0
4,Black or African American Male,0.0
5,Black or African American Female,0.0
6,Asian Male,0.0
7,Asian Female,0.0
8,Native Hawaiian or Other Pacific Islander Female,0.0
9,American Indian or Alaskan Native Female,0.0


In [45]:
calculate_differences(current_year_df, cen_df)

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 24 and the array at index 1 has size 0

In [58]:
cur_df

NameError: name 'cur_df' is not defined